# Try solidfunc

solidfunc is a typed, functional Python library for code-based CAD, built on [SolidCore](https://github.com/tangency-labs/solidcore), a B-rep kernel written in Rust. This notebook builds two small parts. A shape at the end of a cell displays as an interactive 3D view: drag to rotate, scroll to zoom.

Signatures and conventions are in the [API reference](https://github.com/DavidMStraub/solidfunc/blob/main/API.md).

In [ ]:
%pip install --quiet solidfunc

## A plate with a rounded hole

Every operation returns a new shape. Lengths are in millimetres.

In [ ]:
import solidfunc as sf

plate = sf.box(size_x=80, size_y=50, size_z=10)
hole = sf.cylinder(diameter=20, height=10)
part = plate - hole

top = max(sf.faces(part), key=lambda face: face.centroid[2])
hole_edge = [edge for edge in sf.edges(top) if edge.kind == "circle"]
part = sf.fillet(part, hole_edge, radius=2.0)
part

## Selection is plain Python

Faces and edges carry typed data such as `kind`, `area`, `centroid`, `start`, and `end`. Filter them with comprehensions, `max`, or `sorted`.

In [ ]:
for face in sf.faces(part):
    print(f"{face.kind:10} {face.area:9.1f} mm²")

print(f"volume {sf.volume(part):.1f} mm³")

## A clutch brick

The studs are placed on the body's top face, so they follow it when the dimensions change.

In [ ]:
body = sf.box(size_x=15.6, size_y=7.8, size_z=9.6)
top_face = max(sf.faces(body), key=lambda f: f.centroid[2])
top = sf.Plane.from_face(top_face)

stud = sf.cylinder(diameter=4.8, height=1.7)
studs = [sf.place(stud, top.offset(x=x)) for x in (-4, 4)]

brick = sf.union(body, *studs)
brick

## Export

STEP files open in other CAD tools. In Colab, download the file from the file browser on the left.

In [ ]:
sf.export_step(brick, "brick.step")

More examples, rebuilt from the book [CAD as Code](https://github.com/DavidMStraub/cad-as-code-book), are in the [examples folder](https://github.com/DavidMStraub/solidfunc/tree/main/examples). If something breaks or feels wrong, please [open an issue](https://github.com/DavidMStraub/solidfunc/issues).